In [12]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

file_path = r"C:\Users\yildi\Desktop\tamfinans\collection_master_data_v3.csv"
df = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')
df.columns = df.columns.str.strip()

df['invoice_date'] = pd.to_datetime(df['invoice_date'], dayfirst=True, errors='coerce')
df['due_date'] = pd.to_datetime(df['due_date'], dayfirst=True, errors='coerce')

df['tenor'] = (df['due_date'] - df['invoice_date']).dt.days
df = df.dropna(subset=['invoice_date', 'due_date', 'tenor'])

In [13]:
#Data cleaning and outlier detection(Isolation Forest) 
numeric_cols = ['income', 'invoice_amount', 'dpd', 'tenor']
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(df[numeric_cols])
df = df[outliers == 1].copy()

In [14]:
#Label encoding and scaling
le = LabelEncoder()
categorical_cols = ['sector', 'legal_type', 'tax_type', 'factoring_type', 'contact_type']
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [15]:
# PROBLEM 1: Ödeme problemi (XGBoost Classification)

drop_list = ['customer_id', 'customer_name', 'invoice_id', 'invoice_date', 'due_date', 'ptp_status']

X = df.drop(columns=[c for c in drop_list if c in df.columns])
y = df['ptp_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [16]:
# Sonuçlar
y_pred = xgb_model.predict(X_test)

print(f"XGBoost Model Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nDetailed Classification Report:\n", classification_report(y_test, y_pred))

importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importance (Top Features Impacting Model Decisions):")
print(importances)

XGBoost Model Accuracy: 0.74

Detailed Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.76      0.78      1145
           1       0.67      0.71      0.69       755

    accuracy                           0.74      1900
   macro avg       0.73      0.74      0.74      1900
weighted avg       0.75      0.74      0.75      1900


Feature Importance (Top Features Impacting Model Decisions):
dpd               0.482966
sector            0.200801
invoice_amount    0.066560
income            0.065514
legal_type        0.051762
contact_type      0.045683
tax_type          0.044454
factoring_type    0.042261
tenor             0.000000
dtype: float32


In [17]:
# Problem 3: Risk Skor 
from sklearn.ensemble import GradientBoostingClassifier

def define_risk(row):
    if row['ptp_status'] == 0 and row['dpd'] > 15:
        return 2 
    elif row['ptp_status'] == 0 or row['dpd'] > 5:
        return 1  
    else:
        return 0 

df['risk_segment'] = df.apply(define_risk, axis=1)



In [18]:
#sınıf sayısı öğrenme logu

class_counts = df['risk_segment'].nunique()
print(f"Tespit edilen risk sınıfı sayısı: {class_counts}")

Tespit edilen risk sınıfı sayısı: 2


In [19]:
# risk skoru modeli eğitimi
drop_for_risk = ['customer_id', 'customer_name', 'invoice_id', 'invoice_date', 'due_date', 'ptp_status', 'risk_segment', 'risk_score']
X_risk = df.drop(columns=[c for c in drop_for_risk if c in df.columns])
y_risk = df['risk_segment']

X_train_risk, X_test_risk, y_train_risk, y_test_risk = train_test_split(X_risk, y_risk, test_size=0.2, random_state=42)

gb_risk = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
gb_risk.fit(X_train_risk, y_train_risk)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",4
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [20]:
target_index = class_counts - 1
df['risk_score'] = gb_risk.predict_proba(X_risk)[:, target_index] * 100

print("\nRisk Segment Distribution:")
print(df['risk_segment'].value_counts())

print("\nRisk Scores for First 5 Customers:")
print(df[['customer_id', 'risk_score', 'risk_segment']].head())



Risk Segment Distribution:
risk_segment
1    5593
0    3907
Name: count, dtype: int64

Risk Scores for First 5 Customers:
  customer_id  risk_score  risk_segment
0   CUST-1000   67.705611             1
1   CUST-1001   97.035228             1
2   CUST-1002   88.225620             1
3   CUST-1003   99.061611             1
4   CUST-1004   61.026094             0
